For the "Benchmark Test" project, let's analyze all of the test data (stored in an Excel Sheet) and see if we can determine if using a free function is more performant (or worse) than a member function.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# Display all of the rows in a data frame
#pd.set_option('display.max_rows', None)

In [3]:
file_path = 'Vec4_benchmark_results.xlsx'

In [4]:
excel_file = pd.ExcelFile(file_path)
sheet_names = excel_file.sheet_names

In [5]:
# (Note that running this cell may take quite a few seconds e.g 45)

# == Get the testing metadata  ==
metadata_df = pd.concat(
    [pd.read_excel(file_path, sheet_name=sheet, nrows=2, usecols=list(range(7))).assign(**{'Sheet Name': sheet}) for sheet in sheet_names],
    ignore_index=True
)

# Reorder columns to make 'Sheet Name' the first column
cols = metadata_df.columns.tolist()
cols.insert(0, cols.pop(cols.index('Sheet Name')))
metadata_df = metadata_df[cols]


# == Get the Data of the tests itself ==
def extract_test_data_from_sheet(file_path, sheet_name):
    """
    Extracts the testa data from a specific sheet in an Excel file,
    transposes it, and adds a 'Sheet Name' column.

    Args:
        file_path (str): The path to the Excel file.
        sheet_name (str): The name of the sheet to process.

    Returns:
        pandas.DataFrame: A DataFrame containing the transposed data
                          with the first row as headers and an added
                          'Sheet Name' column.
    """
    second_table_df = pd.read_excel(file_path, sheet_name=sheet_name, header=3)
    second_table_df.insert(0, 'Sheet Name', sheet_name)
    return second_table_df

stats_df = pd.concat(
    [extract_test_data_from_sheet(file_path, sheet) for sheet in sheet_names],
    ignore_index=True
)
stats_df.columns.name = None       # Remove that `Unamed: 0` part

#display(metadata_df)
#display(stats_df)

In [6]:
# Put each measurement on a separate row

def melt_data_to_one_measurement_per_row(df, function_str):
    """This function will make it where we have one test run/measurement per row
    with all of the metadata for that test off to the left.  In some of the
    column headers there is meta data that will be "melted" off.  This works on
    a "per function" basis.  Please see some of the sample usage below to fully
    make sense of this.

    Args:
        df (pd.DataFrame): test data to process
        function_str (str): the function to filter on (e.g. `add()`

    Returns:
        pd.DataFrame: the processed data
    """

    # Get a subset that has only the sheet name and columns with the function
    func_only = df.filter(regex=f'{function_str}|Sheet Name')

    # Remove it from the column names
    func_only.columns = func_only.columns.str.replace(f'{function_str}\\(\\)', '', regex=True)

    # Place is after `Sheet Name`
    func_only.insert(1, 'method', function_str)

    # Melt the column `[member]`, `[free][pass_struct]`, `[free][pass_args]` to a new column
    func_only = pd.melt(func_only, id_vars=['Sheet Name', 'method'], var_name='function_type_temp', value_name='time_ns')

    # Add a column called `function_type
    #   if function_type_temp=[member], function_type = `member`
    #   if function_type_temp=[free]..., function_type = `free`
    func_only['function_type'] = np.where(func_only['function_type_temp'].str.contains('member'), 'member', 'free')

    # Add a column called `argument_type`
    #   if function_type_temp is member, then set argument_type=member_vars
    #   if function_type temp contains pass_struct, then set argument_type=struct
    #   if function_type_temp contains pass_args, then set argument_type=args
    func_only['argument_type'] = np.where(func_only['function_type_temp'].str.contains('member'), 'member_vars',
                                          np.where(func_only['function_type_temp'].str.contains('pass_struct'), 'struct',
                                                   np.where(func_only['function_type_temp'].str.contains('pass_args'), 'args', '<ERROR>')
                                                   )
                                          )

    # Drop the `function_type_temp` column
    func_only.drop(columns=['function_type_temp'], inplace=True)

    # Move the time columm to the end
    func_only = func_only[['Sheet Name', 'method', 'function_type', 'argument_type', 'time_ns']]

    return func_only


# Split out each test function into a separate data frame
add_only = melt_data_to_one_measurement_per_row(stats_df, 'add()')
subtract_only = melt_data_to_one_measurement_per_row(stats_df, 'subtract()')
dot_product_only = melt_data_to_one_measurement_per_row(stats_df, 'dot_product()')
normalize_only = melt_data_to_one_measurement_per_row(stats_df, 'normalize()')

#display(add_only)
#display(subtract_only)
#display(dot_product_only)
#display(normalize_only)

# Combine the dataframes back into one
all_measurements_by_sheet = pd.concat([add_only, subtract_only, dot_product_only, normalize_only], ignore_index=True)
#display(all_measurements_by_sheet)

In [7]:
# == Combine each test with its metadata
keep_sheet_name = False

benchmark_data = pd.merge(metadata_df, all_measurements_by_sheet, on='Sheet Name')
benchmark_data.drop(columns=['rng_seed', 'num_runs', 'num_vectors_per_run'], inplace=True)

if not keep_sheet_name:
    benchmark_data.drop(columns=['Sheet Name'], inplace=True)

#display(benchmark_data)

In [8]:
# == Analyze the benchmark stats ==

stats = benchmark_data.copy()
trim_top_by = 0.00              # Percentage
trim_bottom_by = 0.00           # Percentage

# Split all of the rows into separate data frames that have the same column values, but only `time_ns` is different
# Get the column names in the data frame, but don't include `time_ns`
col_names = stats.columns.drop('time_ns')
test_sets = stats[stats.duplicated(subset=col_names, keep=False)].copy()

test_suite_results = []
for _, group in test_sets.groupby(list(col_names)):
    # Sort the group by the time_ns column in ascending order
    sorted_group = group.sort_values(by='time_ns')

    # Optionally, do some trimming, to throw out any major statistical outliers
    #   Note: The desire here is to get rid of any horrible performers (e.g. 30% slower than the median), but when
    #         we also trim the bottom, we're getting rid of the best performing measurement as well.
    num_rows = len(sorted_group)
    trim_top = round(num_rows * trim_top_by)
    trim_bottom = round(num_rows * trim_bottom_by)

    if trim_top:
        sorted_group = sorted_group.iloc[trim_top:]
    if trim_bottom:
        sorted_group = sorted_group.iloc[:-trim_bottom]

    average = int(sorted_group['time_ns'].mean())
    median = int(sorted_group['time_ns'].median())
    fastest = int(sorted_group['time_ns'].min())
    slowest = int(sorted_group['time_ns'].max())
    range = slowest - fastest
    variance = int(sorted_group['time_ns'].var())
    std_dev = int(sorted_group['time_ns'].std())

    # Create a new row with the computed stats
    test_stats = pd.DataFrame({
        'time_ns_mean':     [average],
        'time_ns_median':   [median],
        'time_ns_min':      [fastest],
        'time_ns_max':      [slowest],
        'time_ns_range':    [range],
#         'time_ns_variance': [variance],
#         'time_ns_std_dev':  [std_dev],
    })

    # Values are all in nanoseconds, convert to milliseconds to make it more human readable
    test_stats = (test_stats / 1000000).round(2)

    # Rename all columns with `ns` in them to `ms`
    test_stats.columns = test_stats.columns.str.replace('_ns_', '_ms_')

    # Create a new dataframe, with only the first row from group
    test_results = group.iloc[0].to_frame().T
    test_results.drop(columns=['time_ns'], inplace=True)
    test_results = pd.concat([
        test_results.reset_index(drop=True),
        test_stats.reset_index(drop=True)
    ], axis=1)

    test_suite_results.append(test_results)

# Combine the rows and do a grouping
analyzed_stats = pd.concat(test_suite_results, ignore_index=True)
analyzed_stats_grouped = analyzed_stats.groupby(list(col_names)).first()

#display(analyzed_stats)
#display(analyzed_stats_grouped)

How many test suites were run:

In [9]:
num_test_suites = len(analyzed_stats)
print(num_test_suites)

576


In [10]:
def find_which_is_fastest_and_by_how_much(df, measurement_to_use):
    """This funciton is used to find if using member functions or free functions were faster for a measurement metric"""

    # Only keep the `measurement_to_use` measurement: drop all columns starting with `time_`, but keep that one if it's name is `measurement_to_use`
    meta_data = df.loc[:, ~df.columns.str.startswith('time_')]
    measurement = df[measurement_to_use]
    df = pd.concat([meta_data, measurement], axis=1)
    metadata_cols = list(meta_data.columns)

    # This is how we will compare the data
    comparison_grouping = metadata_cols.copy()
    comparison_grouping.remove('function_type')
    comparison_grouping.remove('argument_type')

    best_runs = []
    grouped = df.groupby(comparison_grouping)
    for _, group in grouped:
        # In each group, look at the `measurement_to_use`, and take the lowest value (the fastest)
        fastest_measurement = group.loc[group[measurement_to_use].idxmin()].to_frame().T
        free_was_faster = (fastest_measurement['function_type'].iloc[0] == 'free')
        member_measurement_value = group[group['function_type'] == 'member'][measurement_to_use].values[0]

        # Figure out the speed difference
        if free_was_faster:
            # If free was faster, get that measurement, and the member function's measurement, and then do the comparison
            free_measurement_value = fastest_measurement[measurement_to_use].values[0]
            by_how_much = (member_measurement_value - free_measurement_value)
        else:
            # Member was faster, there are two measurements for free, lets take the fastest one to put free's best foot forward
            free_measurements = group[group['function_type'] == 'free']
            fastest_free_measurement = free_measurements.loc[free_measurements[measurement_to_use].idxmin()].to_frame().T
            free_measurement_value = fastest_free_measurement[measurement_to_use].values[0]
            by_how_much = (free_measurement_value - member_measurement_value)

        fastest_measurement['time_ms_faster'] = by_how_much
        best_runs.append(fastest_measurement)

    return pd.concat(best_runs)


If you run the above function, you'll see some cases where free is faster and member functions are faster, but if you look at the `time_ms_faster` column, there are a lot of `0.03` and `0.34` entires, meaning that it's pretty much fuzz and not significant at all.  But there are some values are `33.43`, so there are cases with a substantial difference. Let's try to find them.


In [11]:
median_faster_all = find_which_is_fastest_and_by_how_much(analyzed_stats, 'time_ms_median')
average_faster_all = find_which_is_fastest_and_by_how_much(analyzed_stats, 'time_ms_mean')
fastest_all = find_which_is_fastest_and_by_how_much(analyzed_stats, 'time_ms_min')

# Let's filter out for anything that at least has a difference of 10 ms (or 1% of a second)
ms_faster_threshold = 10.0
significant_median_faster = median_faster_all[median_faster_all['time_ms_faster'] >= ms_faster_threshold]
significant_average_faster = average_faster_all[average_faster_all['time_ms_faster'] >= ms_faster_threshold]
#significant_fastest = fastest_all[fastest_all['time_ms_faster'] >= ms_faster_threshold]


display(significant_median_faster)
display(significant_average_faster)
#display(significant_fastest)

,cpu,os,compiler,compilation_flags,method,function_type,argument_type,time_ms_median,time_ms_faster
7,AMD Ryzen 9 6900HX,Ubuntu 24.04,GCC 14.2.0,-O0,normalize(),free,struct,328.73,27.90
67,AMD Ryzen 9 6900HX,Ubuntu 24.04,clang 19.1.1,-O0,normalize(),free,struct,339.41,11.94
90,AMD Ryzen 9 6900HX,Ubuntu 24.04,clang 19.1.1,-O2,normalize(),free,args,205.75,46.50
102,AMD Ryzen 9 6900HX,Ubuntu 24.04,clang 19.1.1,-O3,normalize(),free,args,207.12,37.87
114,AMD Ryzen 9 6900HX,Ubuntu 24.04,clang 19.1.1,-Ofast,normalize(),free,args,206.19,34.13
438,Intel i7-10750H,Ubuntu 24.04,clang 19.1.1,-O2,normalize(),free,args,192.14,40.53
450,Intel i7-10750H,Ubuntu 24.04,clang 19.1.1,-O3,normalize(),free,args,187.26,39.89
462,Intel i7-10750H,Ubuntu 24.04,clang 19.1.1,-Ofast,normalize(),free,args,186.33,43.71


,cpu,os,compiler,compilation_flags,method,function_type,argument_type,time_ms_mean,time_ms_faster
7,AMD Ryzen 9 6900HX,Ubuntu 24.04,GCC 14.2.0,-O0,normalize(),free,struct,328.76,27.87
67,AMD Ryzen 9 6900HX,Ubuntu 24.04,clang 19.1.1,-O0,normalize(),free,struct,339.41,11.95
90,AMD Ryzen 9 6900HX,Ubuntu 24.04,clang 19.1.1,-O2,normalize(),free,args,206.12,46.17
102,AMD Ryzen 9 6900HX,Ubuntu 24.04,clang 19.1.1,-O3,normalize(),free,args,207.15,37.83
114,AMD Ryzen 9 6900HX,Ubuntu 24.04,clang 19.1.1,-Ofast,normalize(),free,args,206.3,34.87
438,Intel i7-10750H,Ubuntu 24.04,clang 19.1.1,-O2,normalize(),free,args,191.9,40.29
450,Intel i7-10750H,Ubuntu 24.04,clang 19.1.1,-O3,normalize(),free,args,186.35,39.69
462,Intel i7-10750H,Ubuntu 24.04,clang 19.1.1,-Ofast,normalize(),free,args,186.04,43.52


So it is interesting to note that there are some cases where a free function is more performant than the member function, and by some margin.  But there's some interesting observations here:

1. The `normalize()` method is exclusively the only method showing up here when we filter, which is a more complex method than others, and also calls the `dot()` method.
2. AMD and Intel environments are the only one that show up in this list
3. This is only happening on Ubuntu Linux, with one exception of Windows
4. clang is the only compiler showing this performance increase
   - with a GCC exception, but sparingly and compiled with `-O0`

But I don't think this means that we should be concluding that free functions are more performant here.  If you look up above we have 576 tests/stats.  Al lot of them have `time_ms_faster` hovering around `0`.  Above, we filtered where it had ot be faster by at least 10 milliseconds.  We only have around 8 cases.  So for 98% of the tests there's no real tangible increase...

And in most cases, that performance bump is only by about 35 ms.  That's very insignificant.

**I don't think there is any significant performance increase (in general) by using a free function vs. a member function**

---

For an epilouge, let's look at two other things.

**No. 1: When writing a free is passing arguments or passing a struct more performant?**

In [12]:
def get_fastest_free_arugment_type(df, measurement_to_use):
    """This funciton is used to find if, when using free functions, is passing a struct or args faster (for a given measurement metric)"""

    # Remove all functions with the type of member
    df = df[df['function_type'] != 'member']

    # Only keep the `measurement_to_use` measurement: drop all columns starting with `time_`, but keep that one if it's name is `measurement_to_use`
    meta_data = df.loc[:, ~df.columns.str.startswith('time_')]
    measurement = df[measurement_to_use]
    df = pd.concat([meta_data, measurement], axis=1)
    metadata_cols = list(meta_data.columns)

    # This is how we will compare the data
    comparison_grouping = metadata_cols.copy()
    comparison_grouping.remove('argument_type')

    best_runs = []
    grouped = df.groupby(comparison_grouping)
    for _, group in grouped:
        # In each group grab which one was faster
        fastest_measurement = group.loc[group[measurement_to_use].idxmin()].to_frame().T
        slowest_measurement = group.loc[group[measurement_to_use].idxmax()].to_frame().T
        by_how_much = slowest_measurement[measurement_to_use].values[0] - fastest_measurement[measurement_to_use].values[0]

        fastest_measurement['time_ms_faster'] = by_how_much
        best_runs.append(fastest_measurement)

    return pd.concat(best_runs)#, ignore_index=True)


median_free = get_fastest_free_arugment_type(analyzed_stats, 'time_ms_median')
average_free = get_fastest_free_arugment_type(analyzed_stats, 'time_ms_mean')
fastest_free = get_fastest_free_arugment_type(analyzed_stats, 'time_ms_min')

# Let's filter out for anything that at least has a difference of 5 ms  (let's use a higher threshold)
ms_faster_threshold = 5.0
median_free_faster = median_free[median_free['time_ms_faster'] >= ms_faster_threshold]
average_free_faster = average_free[average_free['time_ms_faster'] >= ms_faster_threshold]
fastest_free_faster = fastest_free[fastest_free['time_ms_faster'] >= ms_faster_threshold]


# There's actually quite a few appearing here, about 50 rows per table
display(median_free_faster)
display(average_free_faster)
display(fastest_free_faster)

,cpu,os,compiler,compilation_flags,method,function_type,argument_type,time_ms_median,time_ms_faster
7,AMD Ryzen 9 6900HX,Ubuntu 24.04,GCC 14.2.0,-O0,normalize(),free,struct,328.73,42.94
18,AMD Ryzen 9 6900HX,Ubuntu 24.04,GCC 14.2.0,-O1,normalize(),free,args,255.31,31.97
21,AMD Ryzen 9 6900HX,Ubuntu 24.04,GCC 14.2.0,-O1,subtract(),free,args,205.9,8.26
45,AMD Ryzen 9 6900HX,Ubuntu 24.04,GCC 14.2.0,-O3,subtract(),free,args,204.0,6.38
49,AMD Ryzen 9 6900HX,Ubuntu 24.04,GCC 14.2.0,-Ofast,add(),free,struct,204.03,6.61
67,AMD Ryzen 9 6900HX,Ubuntu 24.04,clang 19.1.1,-O0,normalize(),free,struct,339.41,11.04
90,AMD Ryzen 9 6900HX,Ubuntu 24.04,clang 19.1.1,-O2,normalize(),free,args,205.75,46.56
102,AMD Ryzen 9 6900HX,Ubuntu 24.04,clang 19.1.1,-O3,normalize(),free,args,207.12,40.47
114,AMD Ryzen 9 6900HX,Ubuntu 24.04,clang 19.1.1,-Ofast,normalize(),free,args,206.19,34.05
124,AMD Ryzen 9 6900HX,Windows 11 Home,GCC 14.1.0 (w64devkit),-O0,dot_product(),free,struct,511.51,10.88


,cpu,os,compiler,compilation_flags,method,function_type,argument_type,time_ms_mean,time_ms_faster
7,AMD Ryzen 9 6900HX,Ubuntu 24.04,GCC 14.2.0,-O0,normalize(),free,struct,328.76,42.92
18,AMD Ryzen 9 6900HX,Ubuntu 24.04,GCC 14.2.0,-O1,normalize(),free,args,255.23,32.05
21,AMD Ryzen 9 6900HX,Ubuntu 24.04,GCC 14.2.0,-O1,subtract(),free,args,205.97,8.10
45,AMD Ryzen 9 6900HX,Ubuntu 24.04,GCC 14.2.0,-O3,subtract(),free,args,203.97,6.48
49,AMD Ryzen 9 6900HX,Ubuntu 24.04,GCC 14.2.0,-Ofast,add(),free,struct,203.99,6.67
67,AMD Ryzen 9 6900HX,Ubuntu 24.04,clang 19.1.1,-O0,normalize(),free,struct,339.41,11.24
90,AMD Ryzen 9 6900HX,Ubuntu 24.04,clang 19.1.1,-O2,normalize(),free,args,206.12,46.27
102,AMD Ryzen 9 6900HX,Ubuntu 24.04,clang 19.1.1,-O3,normalize(),free,args,207.15,40.42
114,AMD Ryzen 9 6900HX,Ubuntu 24.04,clang 19.1.1,-Ofast,normalize(),free,args,206.3,34.64
124,AMD Ryzen 9 6900HX,Windows 11 Home,GCC 14.1.0 (w64devkit),-O0,dot_product(),free,struct,510.94,9.52


,cpu,os,compiler,compilation_flags,method,function_type,argument_type,time_ms_min,time_ms_faster
7,AMD Ryzen 9 6900HX,Ubuntu 24.04,GCC 14.2.0,-O0,normalize(),free,struct,327.48,42.98
18,AMD Ryzen 9 6900HX,Ubuntu 24.04,GCC 14.2.0,-O1,normalize(),free,args,253.41,33.20
21,AMD Ryzen 9 6900HX,Ubuntu 24.04,GCC 14.2.0,-O1,subtract(),free,args,205.26,7.12
45,AMD Ryzen 9 6900HX,Ubuntu 24.04,GCC 14.2.0,-O3,subtract(),free,args,203.59,6.51
49,AMD Ryzen 9 6900HX,Ubuntu 24.04,GCC 14.2.0,-Ofast,add(),free,struct,203.6,6.63
67,AMD Ryzen 9 6900HX,Ubuntu 24.04,clang 19.1.1,-O0,normalize(),free,struct,338.25,10.80
90,AMD Ryzen 9 6900HX,Ubuntu 24.04,clang 19.1.1,-O2,normalize(),free,args,204.97,45.48
102,AMD Ryzen 9 6900HX,Ubuntu 24.04,clang 19.1.1,-O3,normalize(),free,args,206.29,40.84
114,AMD Ryzen 9 6900HX,Ubuntu 24.04,clang 19.1.1,-Ofast,normalize(),free,args,204.93,34.84
121,AMD Ryzen 9 6900HX,Windows 11 Home,GCC 14.1.0 (w64devkit),-O0,add(),free,struct,470.71,24.67


In [13]:
num_median_free_faster = len(median_free_faster)
num_average_free_faster = len(average_free_faster)
num_fastest_free_faster = len(fastest_free_faster)

print(f'Count where one `argument_type` (for `function_type=free`) was faster (than the other) by {ms_faster_threshold:0.3} ms')
print(f'  Median: {num_median_free_faster}')
print(f'  Average: {num_average_free_faster}')
print(f'  Fastest: {num_fastest_free_faster}')

print('')

# Find how many times `struct` appeared
num_median_struct = len(median_free_faster[median_free_faster['argument_type'] == 'struct'])
num_average_struct = len(average_free_faster[average_free_faster['argument_type'] == 'struct'])
num_fastest_struct = len(fastest_free_faster[fastest_free_faster['argument_type'] == 'struct'])

print(f'Count where `struct` was faster (by {ms_faster_threshold:0.3} ms) ')
print(f'  Median: {num_median_struct}')
print(f'  Average: {num_average_struct}')
print(f'  Fastest: {num_fastest_struct}')

Count where one `argument_type` (for `function_type=free`) was faster (than the other) by 5.0 ms
  Median: 46
  Average: 46
  Fastest: 53

Count where `struct` was faster (by 5.0 ms) 
  Median: 26
  Average: 26
  Fastest: 32


It looks to be that `struct` is faster in approximately 55% of the cases, which means for the inverse (passing arguments) is faster in about 45% of the cases as well.   **I'm hesitant to say there is a significant difference in passing by struct vs passing by argument when freeing a function.**

(As a personal principle of design, I do like passing structures over arguments, especially when the number of arguments grows past 6.)

---

**No. 2: Which compiler did the best on each environment?**
(in each environment)

We'll define Environment as the bundle of a CPU and OS.  I don't want to turn this into a compiler comparison, but it's still fun to look at.

In [14]:
def find_fastest_on_same_compiler(df, measurement_to_use):
    """This function is used to find which suite of test ran the fastest on a given environment.  It operaties by adding
    the runtimes of each function (using the given column)."""

    # Only keep the `measurement_to_use` measurement: drop all columns starting with `time_`, but keep that one if it's name is `measurement_to_use`
    meta_data = df.loc[:, ~df.columns.str.startswith('time_')]
    measurement = df[measurement_to_use]
    df = pd.concat([meta_data, measurement], axis=1)

    metadata_cols = list(meta_data.columns)
    comparison_grouping = metadata_cols.copy()
    comparison_grouping.remove('method')
    sum_column_name = measurement_to_use + '_sum'

    # Figure out how long each compilation took
    best_runs_per_compilation = []
    grouped = df.groupby(comparison_grouping)
    for _, group in grouped:
        # Get a sum of all of the valules in the `measurement_to_use` column
        measurement_sum = group[measurement_to_use].sum()

        # Remove the `method` and timing columsn column from the grouping
        group.drop(columns=['method', measurement_to_use], inplace=True)

        # Reduce the data frame to one row and append on the measurement
        group = group.iloc[0].to_frame().T
        group[sum_column_name] = measurement_sum

        best_runs_per_compilation.append(group)

    # Now let's figure out (per compilation) which one was best
    comparison_grouping.remove('function_type')
    comparison_grouping.remove('argument_type')
    comparison_grouping.remove('compilation_flags')
    best_runs = []
    grouped = pd.concat(best_runs_per_compilation).groupby(comparison_grouping)
    for _, group in grouped:
        fastest_measurement = group.loc[group[sum_column_name].idxmin()].to_frame().T
        best_runs.append(fastest_measurement)

    return pd.concat(best_runs)#, ignore_index=True)

In [15]:
# Let's just first look at the best run for each cpu+os+compiler combo

#median_same_compiler_fastest = find_fastest_on_same_compiler(analyzed_stats, 'time_ms_median')
#average_same_compiler_fastest = find_fastest_on_same_compiler(analyzed_stats, 'time_ms_mean')
fastest_same_compiler_fastest = find_fastest_on_same_compiler(analyzed_stats, 'time_ms_min')

#display(median_same_compiler_fastest)
#display(average_same_compiler_fastest)
display(fastest_same_compiler_fastest)

,cpu,os,compiler,compilation_flags,function_type,argument_type,time_ms_min_sum
50,AMD Ryzen 9 6900HX,Ubuntu 24.04,GCC 14.2.0,-Ofast,member,member_vars,847.25
84,AMD Ryzen 9 6900HX,Ubuntu 24.04,clang 19.1.1,-O2,free,args,812.66
144,AMD Ryzen 9 6900HX,Windows 11 Home,GCC 14.1.0 (w64devkit),-O2,free,args,1485.23
218,AMD Ryzen 9 6900HX,Windows 11 Home,MSVC 19.38.33135,/Ox,member,member_vars,835.23
240,Apple M4,Sequoia 15.6,GCC 15.1.0,-O1,free,args,409.36
301,Apple M4,Sequoia 15.6,clang 17.0.0,-O1,free,struct,487.96
397,Intel i7-10750H,Ubuntu 24.04,GCC 14.2.0,-Ofast,free,struct,636.91
444,Intel i7-10750H,Ubuntu 24.04,clang 19.1.1,-O3,free,args,601.3
494,Intel i7-10750H,Windows 11 Home,GCC 14.1.0 (w64devkit),-O2,member,member_vars,1384.9
564,Intel i7-10750H,Windows 11 Home,MSVC 19.39.33523,/Ox,free,args,562.29


In [16]:
# Now lets see the best on each cpu + os

def find_best_on_each_environment(df):
    # Find column name starting with `time_ms_...`
    measurement = [col for col in df.columns if col.startswith('time_ms_')][0]

    best_runs = []
    grouped = df.groupby(['cpu', 'os'])
    for _, group in grouped:
        # Find the fastest in the group
        fastest = group.loc[group[measurement].idxmin()].to_frame().T
        best_runs.append(fastest)

    return pd.concat(best_runs)

#median_by_envrionment = find_best_on_each_environment(median_same_compiler_fastest)
#average_by_envrionment = find_best_on_each_environment(average_same_compiler_fastest)
fastest_by_envrionment = find_best_on_each_environment(fastest_same_compiler_fastest)

#display(median_by_envrionment)
#display(average_by_envrionment)
display(fastest_by_envrionment)

,cpu,os,compiler,compilation_flags,function_type,argument_type,time_ms_min_sum
84,AMD Ryzen 9 6900HX,Ubuntu 24.04,clang 19.1.1,-O2,free,args,812.66
218,AMD Ryzen 9 6900HX,Windows 11 Home,MSVC 19.38.33135,/Ox,member,member_vars,835.23
240,Apple M4,Sequoia 15.6,GCC 15.1.0,-O1,free,args,409.36
444,Intel i7-10750H,Ubuntu 24.04,clang 19.1.1,-O3,free,args,601.3
564,Intel i7-10750H,Windows 11 Home,MSVC 19.39.33523,/Ox,free,args,562.29


Looking at this, we have a few interesting observations:

- For x86 (64 bit) Linux, clang did better.
- On AMD Ubuntu+clang was the most performant
- on Intel, Windows+MSVC was the fastest
- For Apple, GCC did the best, and with `-O1`!!
  - This is not something that normally happens.  Typically `-O3` is what you want for performance (and `-Ofast` could be better, but is more dangerous)
---

Keep in mind that this benchmark is measuring something very atomic.  It could change in a much larger (complex) program.  And may have different results in the future as compilers (and CPUs) advance.
